# Teacher scoring (milestone 3) -- Llama Guard 3 8B and Qwen3Guard-Gen-8B

Runs `reflex_sentry.teacher.score` over the four processed pools with each guard-model
preset, on a Kaggle GPU T4 x2 notebook. See `reflex_sentry/teacher/score.py` for exactly
what is recorded (first-decision-token verdict probability, plus a category from a short
greedy continuation) and `docs/PLAN.md`'s "Running milestones 3 to 5" section for the
local-side commands that consume the output files this notebook produces.

**Settings -> Accelerator: GPU T4 x2** before running. **Settings -> Internet: on** (needed
to pip install and to download model weights from Hugging Face).


## Getting the repo and data onto Kaggle

Two options, pick one:

**Option A -- private Kaggle Dataset.** Zip the repo (`git archive` or a plain zip) and the
four processed parquets (`data/processed/{train,val_pool,test_pool,test_ood_pool}.parquet`),
upload as a private Kaggle Dataset, and attach it to this notebook (Add Data, on the right
panel). It will appear under `/kaggle/input/<your-dataset-name>/`. This keeps the harmful
prompts in `data/` off any public host, which is required (README section 8: `data/` is
never committed to git and Kaggle Datasets used for it must stay private).

**Option B -- git clone the public repo branch, upload data separately.** If the repo
branch you are running (code only, no `data/`) is public, clone it directly in the cell
below. The processed parquets still need to come from a private Kaggle Dataset (Option A)
since they must not go through a public git remote -- attach just the data as a private
Dataset and clone the code.

Either way, set `REPO_DIR` and `DATA_DIR` in the next cell to wherever your files land.


In [ ]:
%pip install -q "transformers>=4.48" accelerate bitsandbytes


In [ ]:
import os

# --- Option A: private Kaggle Dataset holding a repo zip + the parquets ---
# REPO_DIR = "/kaggle/input/<your-dataset-name>/reflex-sentry"
# DATA_DIR = "/kaggle/input/<your-dataset-name>/data/processed"

# --- Option B: git clone the public code branch, data from a private Dataset ---
# !git clone --branch <branch> --depth 1 https://github.com/<you>/reflex-sentry.git /kaggle/working/reflex-sentry
# REPO_DIR = "/kaggle/working/reflex-sentry"
# DATA_DIR = "/kaggle/input/<your-data-dataset-name>/processed"

REPO_DIR = "/kaggle/working/reflex-sentry"  # EDIT ME
DATA_DIR = "/kaggle/input/reflex-sentry-data/processed"  # EDIT ME

import sys
sys.path.insert(0, REPO_DIR)


## Hugging Face access

`meta-llama/Llama-Guard-3-8B` is gated: visit the model page while logged in to Hugging
Face and accept Meta's license *before* running this notebook, or the download below will
fail with a 403. `Qwen/Qwen3Guard-Gen-8B` is ungated.

Add your HF token as a Kaggle Secret named `HF_TOKEN` (Add-ons -> Secrets) rather than
pasting it in a cell.


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)


## Smoke test

Run each preset with `--limit 20` first and read the printed verdict-token ids and the one
raw generation below for a harmless prompt. Confirm they look right (Llama Guard answers
"safe"/"unsafe\nS<n>"; Qwen3Guard-Gen answers "Safety: Safe|Unsafe|Controversial" then
"Categories: ..."). If a model's actual answer format differs from what is assumed, fix the
corresponding preset's strings in `reflex_sentry/teacher/score.py::PRESETS` (they are kept
isolated there for exactly this reason) before the full run.


Smoke outputs go to `smoke_*.parquet`, separate from the full-run files.


In [ ]:
import os
os.chdir(REPO_DIR)


In [ ]:
cmd = (
    f"python -m reflex_sentry.teacher.score --model llama_guard_3_8b "
    f"--inputs {DATA_DIR}/train.parquet {DATA_DIR}/val_pool.parquet "
    f"{DATA_DIR}/test_pool.parquet {DATA_DIR}/test_ood_pool.parquet "
    f"--out /kaggle/working/smoke_llama_guard_3_8b.parquet "
    f"--batch-size 8 --max-length 512 --load-in-4bit --limit 20"
)
print(cmd)
!{cmd}


In [ ]:
cmd = (
    f"python -m reflex_sentry.teacher.score --model qwen3guard_gen_8b "
    f"--inputs {DATA_DIR}/train.parquet {DATA_DIR}/val_pool.parquet "
    f"{DATA_DIR}/test_pool.parquet {DATA_DIR}/test_ood_pool.parquet "
    f"--out /kaggle/working/smoke_qwen3guard_gen_8b.parquet "
    f"--batch-size 8 --max-length 512 --load-in-4bit --limit 20"
)
print(cmd)
!{cmd}


## Sanity check on val_pool (both presets)

The smoke test above only proves the model loads and answers. This check scores the ~400
val_pool prompts, which carry each source dataset's own label, so you can see whether the
teacher separates unsafe from safe before spending an hour on the full run. Expect
wildguardmix rows with source_label 1 to average well above 0.5, and or_bench and hn_seed
rows to sit low (some hard-negative over-flagging is expected and is itself a finding).
Outputs go to separate `valcheck_*` files so they never mix with the full run.


In [ ]:
cmd = (
    f"python -m reflex_sentry.teacher.score --model llama_guard_3_8b "
    f"--inputs {DATA_DIR}/val_pool.parquet "
    f"--out /kaggle/working/valcheck_llama_guard_3_8b.parquet "
    f"--batch-size 8 --max-length 512 --load-in-4bit"
)
print(cmd)
!{cmd}


In [ ]:
cmd = (
    f"python -m reflex_sentry.teacher.score --model qwen3guard_gen_8b "
    f"--inputs {DATA_DIR}/val_pool.parquet "
    f"--out /kaggle/working/valcheck_qwen3guard_gen_8b.parquet "
    f"--batch-size 8 --max-length 512 --load-in-4bit"
)
print(cmd)
!{cmd}


In [ ]:
import pandas as pd

pool = pd.read_parquet(f"{DATA_DIR}/val_pool.parquet")
for model in ["llama_guard_3_8b", "qwen3guard_gen_8b"]:
    path = f"/kaggle/working/valcheck_{model}.parquet"
    if not os.path.exists(path):
        print(model, "not scored yet")
        continue
    m = pool.merge(pd.read_parquet(path), on="id")
    print(f"== {model}: {len(m)} rows")
    print(m.groupby(["source", "source_label"], dropna=False).p_unsafe_teacher
            .describe()[["count", "mean", "50%"]].round(3))
    print("first word of raw answers:", m.teacher_raw.str.split().str[0].value_counts().head().to_dict())


## Full run

Only proceed once the smoke test above printed a sane verdict split and a sane raw
generation for both presets. Drop `--limit`; this scores every id in the four processed
pools (tens of thousands of short prompts take roughly an hour or two on a T4, per README
section 3). `--load-in-4bit` fits an 8B model on one 16GB T4; if you have both T4s and
enough VRAM without 4-bit, `--device-map auto` in fp16 is faster instead -- use one or the
other, not both.


In [ ]:
cmd = (
    f"python -m reflex_sentry.teacher.score --model llama_guard_3_8b "
    f"--inputs {DATA_DIR}/train.parquet {DATA_DIR}/val_pool.parquet "
    f"{DATA_DIR}/test_pool.parquet {DATA_DIR}/test_ood_pool.parquet "
    f"--out /kaggle/working/teacher_scores_llama_guard_3_8b.parquet "
    f"--batch-size 8 --max-length 512 --load-in-4bit --checkpoint-every 10"
)
print(cmd)
!{cmd}


In [ ]:
cmd = (
    f"python -m reflex_sentry.teacher.score --model qwen3guard_gen_8b "
    f"--inputs {DATA_DIR}/train.parquet {DATA_DIR}/val_pool.parquet "
    f"{DATA_DIR}/test_pool.parquet {DATA_DIR}/test_ood_pool.parquet "
    f"--out /kaggle/working/teacher_scores_qwen3guard_gen_8b.parquet "
    f"--batch-size 8 --max-length 512 --load-in-4bit --checkpoint-every 10"
)
print(cmd)
!{cmd}


## Saving outputs

Both `teacher_scores_*.parquet` files are already written under `/kaggle/working/`, which
Kaggle keeps as this notebook's Output tab -- download them from there into your local
`data/interim/` (do **not** commit them; `data/` stays out of git per README section 8).
Then locally:

```bash
python -m reflex_sentry.teacher.as_predictor \
    --scores data/interim/teacher_scores_llama_guard_3_8b.parquet \
    --eval data/processed/val.parquet \
    --out preds/teacher_llama_guard_3_8b_val.csv

python -m reflex_sentry.targets \
    --teachers data/interim/teacher_scores_*.parquet \
    --pool data/processed/train.parquet \
    --out data/interim/soft_targets.parquet
```


In [ ]:
import pandas as pd

for name in ["teacher_scores_llama_guard_3_8b", "teacher_scores_qwen3guard_gen_8b"]:
    path = f"/kaggle/working/{name}.parquet"
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(name, len(df), "rows")
        display(df.head())
    else:
        print(name, "not found -- run the full-run cell above first")
